# Engine v3 · Sunfish — play it through the harness

Sunfish is always White. Clone the engine into the portfolio root (the folder with `scripts/`, `games/` and `engines/`) first:

```bash
git clone https://github.com/thomasahle/sunfish.git
```

Open the notebook from the portfolio root or from `notebooks/` — the settings cell moves to the root either way, so `sunfish/` and `games/` are found.

Set the opponent in the settings cell: `"you"` (type moves in SAN — Nf6, exd5, O-O), `"stockfish"` (Stockfish 16 at the Elo you set), `"greedy"` (the Setup-Week bot) or `"random"`. `MOVES` replays Black's moves so a game you played at the keyboard is reproduced when the notebook is re-run.

In [1]:
%pip install chess -q
OPPONENT  = "stockfish"     # "you" | "stockfish" | "greedy" | "random"
SECONDS   = 0.2             # Sunfish's clock per move -- try 0.2, then 2.0
DEPTH     = None            # or a fixed depth instead of a clock, e.g. 4
ELO       = 1350            # Stockfish's rating limit
MOVES     = ""              # Black's moves to replay, in SAN, e.g. "e5 Nc6 Nf6" (opponent "you")
SUNFISH   = "sunfish/sunfish.py"
MAX_MOVES = 150

import os
if not os.path.isdir("scripts") and os.path.isdir(os.path.join("..", "scripts")):
    os.chdir("..")                      # opened from notebooks/ -- work from the portfolio root instead
print("working in", os.path.basename(os.getcwd()))


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
working in content


## The pieces: finding Stockfish, a keyboard mover, the Setup-Week greedy bot

In [2]:
import datetime
import os
import random
import shutil
import sys
import time

import chess
import chess.engine
import chess.pgn


def find_stockfish():
    for p in [os.environ.get("STOCKFISH_PATH"), shutil.which("stockfish"), "/usr/games/stockfish",
              "/usr/local/bin/stockfish", "/opt/homebrew/bin/stockfish", "./stockfish", "./stockfish.exe"]:
        if p and os.path.isfile(p):
            return p
    return None


def human_mover(scripted):
    """Black = you: typed at the keyboard, or replayed from --moves until the list runs out."""
    queue = list(scripted)

    def move(board):
        while True:
            text = queue.pop(0) if queue else input(f"  your move ({board.fullmove_number}... SAN like Nf6): ").strip()
            for parse in (board.parse_san, board.parse_uci):
                try:
                    return parse(text)
                except ValueError:
                    pass
            print("  not legal here -- try again")
    return move


def greedy_bot(board):
    """The Setup-Week baseline: one ply, take the move that leaves the best material, random tie-break."""
    values = {chess.PAWN: 100, chess.KNIGHT: 280, chess.BISHOP: 320, chess.ROOK: 479, chess.QUEEN: 929, chess.KING: 0}
    def material(b):
        s = sum(values[p.piece_type] * (1 if p.color == chess.WHITE else -1) for p in b.piece_map().values())
        return s if b.turn == chess.WHITE else -s
    best, moves = None, []
    for m in board.legal_moves:
        board.push(m); s = 10**6 if board.is_checkmate() else -material(board); board.pop()
        if best is None or s > best: best, moves = s, [m]
        elif s == best: moves.append(m)
    return random.choice(moves)


## The game loop — `play_sunfish()`

The same loop as `play_andoma()`; only the launch line and the report after each move change. `result.info` carries Sunfish's `info depth … nodes … score …` line, which python-chess parses for you.

In [4]:
from IPython.display import display, clear_output

def play_sunfish(opponent="you", seconds=0.2, depth_limit=None, elo=1350, moves="",
                 sunfish=os.path.join("sunfish", "sunfish.py"), max_moves=150):
    """Sunfish (White) against opponent -- 'you', 'stockfish', 'greedy' or 'random'.  Prints every move with the depth,
    nodes and score Sunfish reports, then the final board; writes games/v3-sunfish-vs-<opponent>.pgn."""
    if not os.path.isfile(sunfish):
        raise SystemExit(f"{sunfish} not found -- clone it first: git clone https://github.com/thomasahle/sunfish.git")
    sunfish_dir = os.path.dirname(os.path.abspath(sunfish))
    engine = chess.engine.SimpleEngine.popen_uci([sys.executable, os.path.basename(sunfish)], cwd=sunfish_dir)
    print("engine.id =", engine.id)                                  # the proof it answered

    sf = None
    if opponent == "stockfish":
        path = find_stockfish()
        if not path:
            raise SystemExit("Stockfish not found -- set STOCKFISH_PATH or put it on your PATH")
        sf = chess.engine.SimpleEngine.popen_uci(path)
        opt = sf.options["UCI_Elo"]
        elo = max(opt.min, min(opt.max, elo))
        sf.configure({"UCI_LimitStrength": True, "UCI_Elo": elo})
        opponent_move = lambda b: sf.play(b, chess.engine.Limit(time=0.1)).move
        opponent_name = f"Stockfish ({elo})"
    elif opponent == "greedy":
        random.seed(1)
        opponent_move, opponent_name = greedy_bot, "greedy bot (one ply)"
    elif opponent == "random":
        random.seed(1)
        opponent_move, opponent_name = (lambda b: random.choice(list(b.legal_moves))), "random mover"
    else:
        opponent_move, opponent_name = human_mover(moves.split()), "me"

    limit = chess.engine.Limit(depth=depth_limit) if depth_limit else chess.engine.Limit(time=seconds)
    setting = f"depth {depth_limit}" if depth_limit else f"{seconds:g} s a move"
    print(f"White: Sunfish ({setting})    Black: {opponent_name}\n")

    board, t_start, total_nodes, stopped = chess.Board(), time.time(), 0, None
    try:
        while not board.is_game_over(claim_draw=True) and board.fullmove_number <= max_moves:
            t0 = time.time()
            if board.turn == chess.WHITE:
                result = engine.play(board, limit, info=chess.engine.INFO_ALL)
                move, who = result.move, "SUN"
                depth, nodes = result.info.get("depth", "?"), result.info.get("nodes", 0)
                score = result.info.get("score")
                cp = score.white().score(mate_score=100000) if score is not None else None
                total_nodes += nodes or 0
                note = f"  depth {depth:>2}  {nodes:>7,} nodes  score {cp:+d}" if cp is not None else ""
            else:
                move, who, note = opponent_move(board), "opp", ""
            num = f"{board.fullmove_number}." if board.turn == chess.WHITE else "   "
            san = board.san(move)
            board.push(move)

            # Print the move progress logs
            print(f"  {num:>4} {who} {san:<8} ({time.time() - t0:4.1f}s){note}")

    except KeyboardInterrupt:
        stopped = "stopped by hand"
    finally:
        engine.quit()
        if sf:
            sf.quit()

    over = board.is_game_over(claim_draw=True)
    outcome = board.outcome(claim_draw=True)
    print(f"\n{len(board.move_stack)} plies in {time.time() - t_start:.0f}s.  Result "
          f"{board.result(claim_draw=True) if over else '*'}  "
          f"({outcome.termination.name.lower() if outcome else 'unfinished'})   "
          f"Sunfish searched {total_nodes:,} nodes in all.")

    game = chess.pgn.Game.from_board(board)
    game.headers.update({"Event": "Engine v3", "Site": "Toledo, OH",
                         "Date": datetime.date.today().strftime("%Y.%m.%d"), "Round": "-",
                         "White": f"Sunfish ({setting})", "Black": opponent_name,
                         "Result": board.result(claim_draw=True) if over else "*"})
    if stopped:
        game.headers["Termination"] = stopped
    os.makedirs("games", exist_ok=True)
    path = os.path.join("games", f"v3-sunfish-vs-{'me' if opponent == 'you' else opponent}.pgn")
    with open(path, "w") as f:
        print(game, file=f, end="\n\n")
    print("saved game ->", path)
    return board

In [5]:
!sudo apt-get update && sudo apt-get install -y stockfish

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,083 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,411 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.4 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Ge

In [6]:
!git clone https://github.com/thomasahle/sunfish.git

Cloning into 'sunfish'...
remote: Enumerating objects: 8330, done.
remote: Counting objects: 100% (1095/1095), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 8330 (delta 906), reused 849 (delta 839), pack-reused 7235 (from 3)
Receiving objects: 100% (8330/8330), 17.01 MiB | 23.17 MiB/s, done.
Resolving deltas: 100% (5788/5788), done.


## Play

Against you: set `OPPONENT = "you"` and answer the prompt in SAN. Against Stockfish: as set. The PGN lands in `games/`.

In [10]:
board = play_sunfish(opponent="you", seconds=SECONDS, depth_limit=DEPTH, elo=ELO, moves=MOVES, sunfish=SUNFISH, max_moves=MAX_MOVES)

engine.id = {'name': 'sunfish 2026'}
White: Sunfish (0.2 s a move)    Black: me

    1. SUN f4       ( 0.2s)  depth  2    1,885 nodes  score +17

1 plies in 2s.  Result *  (unfinished)   Sunfish searched 1,885 nodes in all.
saved game -> games/v3-sunfish-vs-me.pgn


In [8]:
board = play_sunfish(opponent="stockfish", seconds=SECONDS, depth_limit=DEPTH, elo=ELO, moves=MOVES, sunfish=SUNFISH, max_moves=MAX_MOVES)

engine.id = {'name': 'sunfish 2026'}
White: Sunfish (0.2 s a move)    Black: Stockfish (1350)

    1. SUN f4       ( 0.3s)  depth  2    1,885 nodes  score +17
       opp d5       ( 0.1s)
    2. SUN d4       ( 0.2s)  depth  2      948 nodes  score +4
       opp a6       ( 0.1s)
    3. SUN e3       ( 0.2s)  depth  2    1,156 nodes  score +9
       opp Nf6      ( 0.1s)
    4. SUN Bd3      ( 0.2s)  depth  3      959 nodes  score +28
       opp e6       ( 0.1s)
    5. SUN Bd2      ( 0.3s)  depth  3      837 nodes  score +21
       opp Bd6      ( 0.1s)
    6. SUN Nf3      ( 0.2s)  depth  3      932 nodes  score +15
       opp c5       ( 0.1s)
    7. SUN Ne5      ( 0.3s)  depth  2    1,927 nodes  score +25
       opp O-O      ( 0.1s)
    8. SUN O-O      ( 0.2s)  depth  2    1,041 nodes  score +23
       opp Nc6      ( 0.1s)
    9. SUN Nxc6     ( 0.1s)  depth  2      841 nodes  score +20
       opp bxc6     ( 0.1s)
   10. SUN dxc5     ( 0.3s)  depth  3      362 nodes  score -6
       opp Bc7  

## The clock experiment (Task 2.3) — what does more time buy?

Ask Sunfish for its first move from a middlegame position at 0.2 s, 1 s and 3 s. Depth, nodes and sometimes the move itself change; that is iterative deepening at work.

In [9]:
engine = chess.engine.SimpleEngine.popen_uci([sys.executable, os.path.basename(SUNFISH)], cwd=os.path.dirname(os.path.abspath(SUNFISH)))
position = chess.Board("r1bqk2r/pppp1ppp/2n5/8/1bBPn3/2N2N2/PP3PPP/R1BQK2R w KQkq - 0 8")   # Week 4's Italian middlegame
for seconds in (0.2, 1.0, 3.0):
    t0 = time.time()
    r = engine.play(position, chess.engine.Limit(time=seconds), info=chess.engine.INFO_ALL)
    cp = r.info["score"].white().score(mate_score=100000)
    print(f"{seconds:>4} s: {position.san(r.move):<6} depth {r.info.get('depth', '?'):>2}  {r.info.get('nodes', 0):>7,} nodes  score {cp:+d}   ({time.time() - t0:.1f}s)")
engine.quit()


 0.2 s: Qb3    depth  3    2,228 nodes  score -38   (0.2s)
 1.0 s: Qb3    depth  4    6,009 nodes  score -42   (1.1s)
 3.0 s: Qb3    depth  5   12,864 nodes  score +11   (2.0s)
